---
# 2. Loading of Dataset

In this section, we will be loading the EMNIST Dataset provided from the Project Brief. There will only be a Training Data and Validation Data instead of the convention 3 sets (Training, Validation and Test). This is because the GAN is expected to generate and improve it's data through the usage of the Training Data only. Data Augmentation will also be conducted so as to theoretically allow the GAN and VAE Models to generalise better.

---
## 2.1 Preliminary Loading of Dataset

In this sub-section, we will be loading in the Dataset provided and conduct a preliminary observation of 1 image for each Sample Class. Subsequently, we will have a decision on whether there is a need to re-import the data due to any data errors or incorrect classes. The Dataset will be loaded in the Code Cell below.

In [ ]:
# ========== Import Raw Dataset ========== #
df = pd.read_csv('emnist-letters-train.csv')

# ========== Split Labels and Features ========== #
y = df.iloc[:, 0].values
X = df.iloc[:, 1:].values

# ========== Reshape to (28, 28, 1) ========== #
X = X.reshape(-1, 28, 28, 1).astype('float32')

# ========== Normalise Pixel Values ========== #
X = X / 127.5 - 1.0

# ========== One-Hot Encoding (26 Classes) ========== #
y_cat = to_categorical(y - 1, num_classes=26)

# ========== Train/Val Split ========== #
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)
y_cat_train, y_cat_val = train_test_split(
    y_cat, test_size=0.1, stratify=y, random_state=42
)

# ========== Plot One Sample Per Class ========== #
plt.figure(figsize=(12, 6))
all_labels = sorted(np.unique(y_train))

for idx, label in enumerate(all_labels):
    sample_idx = np.where(y_train == label)[0][0]
    image = X_train[sample_idx]

    plt.subplot(3, 9, idx + 1)
    plt.imshow(image.squeeze(), cmap='gray')
    plt.title(f"{label}", fontsize=12)
    plt.axis('off')

plt.suptitle("One Sample Image per Class (All Labels)", fontsize=16, y=1)
plt.tight_layout()
plt.show()

With reference to the output of the Code Cell above, we are able to determine that there is a need to re-import the data and filter accordingly as there are illegitimate data (-2 and -1). Other Data issues are also observed which will be addressed later on.

---
## 2.2 Data Loading & Data Pre-Processing

In this sub-section, we will be loading the Provided EMNIST Dataset. We will also be Spliting the Labels and Features, Reshaping, Normalising, One-Hot Encoding, and Train-Val Split. A detailed summary of the operations that will be conducted in this sub-section have been indicated below:

- Loading of EMNIST CSV Dataset (By Module Coordinator).
- Reshaping into 2D Images with 1 Channel (28, 28, 1).
- Normalise Pixel to [0, 1].
- One-Hot Encoding
- Train / Validation Split (90/10)

With the summary of the operations indicated, we will proceed to conduct the various operations in the Code Cell below.

In [ ]:
# ========== Import Raw Dataset ========== #
df = pd.read_csv('emnist-letters-train.csv')

# ========== Split Labels and Features ========== #
y = df.iloc[:, 0].values
X = df.iloc[:, 1:].values

# ========== Reshape to (28, 28, 1) ========== #
X = X.reshape(-1, 28, 28, 1).astype('float32')

# ========== Normalise Pixel Values ========== #
X = X / 127.5 - 1.0

# ========== Filter to 16 Selected Labels (Original Values) ========== #
selected_labels = [1, 2, 4, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 24, 26]  # A, B, D, E, F, G, I, J, L, N, O, P, Q, T, X, Z
mask = np.isin(y, selected_labels)
X = X[mask]
y = y[mask]

# ========== One-Hot Encoding (26 Classes) ========== #
y_cat = to_categorical(y - 1, num_classes=26)

# ========== Train/Val Split ========== #
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
y_cat_train, y_cat_val = train_test_split(y_cat, test_size=0.1, stratify=y, random_state=42)

# ========== Create Shape Summary DataFrame ========== #
shape_summary = pd.DataFrame({
    'Dataset': ['Original (df)', 'Filtered X', 'Filtered y', 'X_train', 'X_val', 'y_train', 'y_val', 'y_cat_train', 'y_cat_val'],
    'Shape': [df.shape, X.shape, y.shape, X_train.shape, X_val.shape, y_train.shape, y_val.shape, y_cat_train.shape, y_cat_val.shape]
})

# ========== Display Shape Summary ========== #
shape_summary.style.background_gradient(cmap="Blues")

With reference to the output above, we are able to identify that we have successfully imported the Provided EMNIST Dataset. With this observation, we will be able to move on to the Augmenting the Training Data to allow for Comparison between Augmented and Non-Augmented Training Data which theoretically will make the GAN and VAE generalise better. However, theoretical knowledge regarding Augmented Data need not be true and can be situational depending on context and other elements such as Model Architecture and Augmentation Strategy.

---
## 2.3 Data Augmentation

In this sub-section, we will be performing Data Augmentation on the Training Data which was the result of the Train-Validation Split done previously. Data Augmentation will theoretically improve the Model's ability to generalise but subjected to various context and circumstances. The various Augmentation that will be conducted are indicated below:

- Rotation: 15
- Width Shift: Max 10%
- Height Shift: Max 10%
- Zoom: 0.1

With the Augmentation Parameters included, we will proceed to conduct the Data Augmentation on the Training Data in the Code Cell below. We will also set the Seed to 42 to allow for replication.

In [ ]:
# ========== Set Random Seed for Reproducibility ========== #
SEED = 42

# ========== Set Up ImageDataGenerator ========== #
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

# ========== Fit the Generator to Training Data ========== #
datagen.fit(X_train, seed=SEED)

# ========== Create Augmented Training Generator ========== #
augmented_generator = datagen.flow(
    X_train, y_train, batch_size=32, shuffle=True, seed=SEED
)

# ========== Final Confirmation ========== #
print("Data Augmentation Pipeline Set Up Successfully.")

With reference to the output of the Code Cell above, we are able to confirm that the Data Augmentation have been executed successfully and we are able to proceed with Exploratory Data Analysis on our EMNIST Dataset in the next Section.